# Проверка исходных данных F1

Диагностический ноутбук этапа 1. Источник — PostgreSQL проекта, не сохранённая CSV-выборка. Запускайте в окружении проекта после `f1-pipeline collect`.

Одна строка laps — круг пилота в сессии. Итоговые модели ещё не обучались.

In [1]:
from f1_pipeline.quality import inspect_database
from f1_pipeline.db import connect
from f1_pipeline.config import ROOT
from IPython.display import display, HTML
import html

def table(rows):
    if not rows:
        return
    keys = list(rows[0])
    markup = "<table><thead><tr>" + "".join("<th>"+html.escape(str(k))+"</th>" for k in keys) + "</tr></thead><tbody>"
    for row in rows:
        markup += "<tr>" + "".join("<td>"+html.escape(str(row[k]))+"</td>" for k in keys) + "</tr>"
    display(HTML(markup+"</tbody></table>"))

report = inspect_database()

## Покрытие и пропуски

Счётчики отражают исходные данные. NULL времени круга не заменяется нулём. Число записей пилот–сессия отличается от числа уникальных людей.

In [2]:
table(report["sessions"])

session_key,year,circuit_short_name,date_start,drivers,laps,stints,weather,race_control,null_duration,null_start,pit_out,rain_samples,rain_unknown
9102,2023,Catalunya,2023-06-04 13:00:00+00:00,20,1312,63,154,49,2,2,43,0,0
9539,2024,Catalunya,2024-06-23 13:00:00+00:00,20,1310,62,154,107,1,1,42,1,0
9971,2025,Catalunya,2025-06-01 13:00:00+00:00,19,1204,73,155,102,2,0,54,0,0


## Связи и погодное покрытие

Погода сопоставляется последним наблюдением с временем не позже начала круга. Круги без отрезка шин потребуют решения на этапе 2.

In [3]:
table(report["join_coverage"])
table(report["numeric_profiles"]["weather.track_temperature"])

year,laps,unmatched_stint,multiple_stints,missing_prior_weather,max_weather_age_seconds
2023,1312,0,0,0,59.999000
2024,1310,0,0,0,59.882000
2025,1204,2,0,0,59.948000


year,n,missing,minimum,maximum,mean,median,mode,variance,stddev
2023,154,0,26.8,42.5,33.83896103896103,31.75,29.9,25.10853747559634,5.010841992679109
2024,154,0,38.3,43.6,41.09610389610389,41.150000000000006,41.8,1.1219455054749194,1.0592192905507902
2025,155,0,46.5,50.4,48.543225806451574,48.7,49.4,0.9719505655634937,0.9858755324905338


## Проверка SQL-извлечения

Целевая переменная — lap_duration. Здесь ещё присутствуют круги, которые будут исключены при подготовке. Проверяется направление временного объединения, а не готовность модели.

In [4]:
with connect() as conn:
    candidates = conn.execute((ROOT / "sql/003_extract.sql").read_text()).fetchall()
assert all(r["weather_observed_at"] is None or r["weather_observed_at"] <= r["date_start"] for r in candidates)
print("Извлечено строк:", len(candidates))
table(candidates[:5])

Извлечено строк: 3826


session_key,driver_number,full_name,lap_number,date_start,lap_duration,is_pit_out_lap,stint_number,compound,tyre_age_before_lap,weather_observed_at,air_temperature,track_temperature,rainfall
9102,1,Max VERSTAPPEN,1,2023-06-04 13:03:15.969000+00:00,84.281,False,1,MEDIUM,0,2023-06-04 13:02:27.720000+00:00,22.6,33.7,0
9102,1,Max VERSTAPPEN,2,2023-06-04 13:04:40.249000+00:00,80.402,False,1,MEDIUM,1,2023-06-04 13:04:27.749000+00:00,22.6,33.4,0
9102,1,Max VERSTAPPEN,3,2023-06-04 13:06:00.560000+00:00,80.499,False,1,MEDIUM,2,2023-06-04 13:05:27.748000+00:00,22.5,33.3,0
9102,1,Max VERSTAPPEN,4,2023-06-04 13:07:21.199000+00:00,80.346,False,1,MEDIUM,3,2023-06-04 13:06:27.747000+00:00,22.4,33.0,0
9102,1,Max VERSTAPPEN,5,2023-06-04 13:08:41.510000+00:00,80.283,False,1,MEDIUM,4,2023-06-04 13:08:27.745000+00:00,22.4,32.6,0


## Проверки источника и вывод

Исправления и фильтрация пока не выполнялись. Недостающие тайминги и дождевой замер требуют обработки на следующем этапе. Три гонки ограничивают независимое разнообразие погодных условий.

In [5]:
import json
assert all(b["hash_matches"] for b in report["batches"])
assert not any(report["duplicates"].values())
table([{ "severity": i["severity"], "issue": i["issue"], "details": json.dumps({k:v for k,v in i.items() if k not in ("severity", "issue")}, ensure_ascii=False, default=str)} for i in report["issues"]])

severity,issue,details
medium,"Null lap timing; preserve now, exclude or resolve in stage 2","{""year"": 2023, ""null_duration"": 2, ""null_start"": 2}"
medium,"Null lap timing; preserve now, exclude or resolve in stage 2","{""year"": 2024, ""null_duration"": 1, ""null_start"": 1}"
medium,Dry scope needs temporal rainfall filtering,"{""year"": 2024, ""rain_samples"": 1, ""rain_unknown"": 0}"
medium,"Null lap timing; preserve now, exclude or resolve in stage 2","{""year"": 2025, ""null_duration"": 2, ""null_start"": 0}"
high,Join coverage requires stage-2 treatment,"{""year"": 2025, ""laps"": 1204, ""unmatched_stint"": 2, ""multiple_stints"": 0, ""missing_prior_weather"": 0, ""max_weather_age_seconds"": ""59.948000""}"
